# 02 · SQL Exploration & Extraction

Before touching pandas, we use **plain SQL** to understand what is in the database: row counts, missing values, junk values, duplicates and inconsistent formats. Then we extract the full table to `data/raw/raw_customers.csv`.

## 1. Imports and connection

In [1]:
import sqlite3

import pandas as pd

DB_PATH = "../data/raw/advertising.db"
RAW_CSV = "../data/raw/raw_customers.csv"

conn = sqlite3.connect(DB_PATH)
print(f"Connected to {DB_PATH} ✓")

Connected to ../data/raw/advertising.db ✓


## 2. Basic exploration with SQL

In [2]:
# How many rows in total?
total = pd.read_sql("SELECT COUNT(*) AS total FROM customers", conn)
print("Total rows:")
print(total)

# How many customers per country? (brace yourself)
by_country = pd.read_sql("""
    SELECT country, COUNT(*) AS count
    FROM customers
    GROUP BY country
    ORDER BY count DESC
""", conn)
print("\nCustomers per country (unique values in the column):")
print(by_country.to_string())

Total rows:
   total
0    550

Customers per country (unique values in the column):
      country  count
0      Mexico     99
1    Colombia     73
2   Argentina     69
3       Chile     52
4      mexico     39
5    colombia     29
6        Peru     24
7                 24
8       Other     23
9          MX     23
10  argentina     17
11        COL     15
12         PE     13
13      chile     11
14         CL     11
15     MEXICO     10
16       Perú      9
17        ARG      9


## 3. Detect missing / empty values

In this table empty values are `''` (empty strings), not `NULL`, because everything was stored as `TEXT` — very common in real-world CSV exports.

In [3]:
empty = pd.read_sql("""
    SELECT
        SUM(CASE WHEN age           = '' OR age           IS NULL THEN 1 ELSE 0 END) AS age_empty,
        SUM(CASE WHEN gender        = '' OR gender        IS NULL THEN 1 ELSE 0 END) AS gender_empty,
        SUM(CASE WHEN country       = '' OR country       IS NULL THEN 1 ELSE 0 END) AS country_empty,
        SUM(CASE WHEN channel       = '' OR channel       IS NULL THEN 1 ELSE 0 END) AS channel_empty,
        SUM(CASE WHEN monthly_spend = '' OR monthly_spend IS NULL THEN 1 ELSE 0 END) AS spend_empty,
        SUM(CASE WHEN clicks        = '' OR clicks        IS NULL THEN 1 ELSE 0 END) AS clicks_empty
    FROM customers
""", conn)
print("Empty values per column:")
print(empty.to_string())

Empty values per column:
   age_empty  gender_empty  country_empty  channel_empty  spend_empty  clicks_empty
0         48            27             24             23           92            21


## 4. Detect junk values

`N/A`, `none`, `unknown`, `??` — they are not `NULL`, but they are not data either.

In [4]:
junk = pd.read_sql("""
    SELECT
        SUM(CASE WHEN age           IN ('N/A','none','unknown','??') THEN 1 ELSE 0 END) AS age_junk,
        SUM(CASE WHEN gender        IN ('N/A','none','unknown','??') THEN 1 ELSE 0 END) AS gender_junk,
        SUM(CASE WHEN monthly_spend IN ('N/A','none','unknown','??') THEN 1 ELSE 0 END) AS spend_junk,
        SUM(CASE WHEN clicks        IN ('N/A','none','unknown','??') THEN 1 ELSE 0 END) AS clicks_junk
    FROM customers
""", conn)
print("Junk values per column:")
print(junk.to_string())

Junk values per column:
   age_junk  gender_junk  spend_junk  clicks_junk
0        15           23          86            0


## 5. Detect duplicates

Same age, gender, country, channel and spend → most likely the same customer.

In [5]:
duplicates = pd.read_sql("""
    SELECT age, gender, country, channel, monthly_spend, COUNT(*) AS times
    FROM customers
    GROUP BY age, gender, country, channel, monthly_spend
    HAVING COUNT(*) > 1
    ORDER BY times DESC
    LIMIT 15
""", conn)
print(f"Duplicate groups found: {len(duplicates)}")
print(duplicates.to_string())

Duplicate groups found: 15
   age  gender    country    channel monthly_spend  times
0                     ARG    Tik Tok   2256.43 USD      2
1               Argentina   Facebook          4978      2
2            F     Mexico   facebook     $2,904.91      2
3       Female      Chile  Instagram           N/A      2
4         Male         CL  Instagram           N/A      2
5          Man     Mexico         FB                    2
6        Other             Instagram       $955.74      2
7        Other      Chile   Facebook     $2,694.76      2
8        Other     Mexico         FB       3139.75      2
9   19  Female     mexico  instagram       4760.85      2
10  19   Woman   Colombia     Google     $4,144.46      2
11  23     Man                tiktok          4502      2
12  24    male   Colombia     Google     $2,723.63      2
13  27       F      chile  Instagram        757.06      2
14  28    Male   Colombia  Instagram     $4,444.13      2


## 6. Inspect the different `monthly_spend` formats

In [6]:
spend_formats = pd.read_sql("""
    SELECT monthly_spend, COUNT(*) AS times
    FROM customers
    WHERE monthly_spend != '' AND monthly_spend NOT IN ('N/A','none')
    GROUP BY monthly_spend
    ORDER BY times DESC
    LIMIT 20
""", conn)
print("Sample of different formats in monthly_spend:")
print(spend_formats.to_string())

Sample of different formats in monthly_spend:
   monthly_spend  times
0     872.30 USD      2
1         811.14      2
2            773      2
3         757.06      2
4     614.40 USD      2
5           4978      2
6        4760.85      2
7           4502      2
8        4387.52      2
9    4024.72 USD      2
10          3866      2
11   3297.68 USD      2
12       3139.75      2
13        296.74      2
14          2324      2
15       2296.22      2
16   2256.43 USD      2
17   2150.16 USD      2
18   2028.79 USD      2
19          1926      2


## 7. Extract everything into pandas

A single query brings everything; pandas handles the rest.

In [7]:
df = pd.read_sql("SELECT * FROM customers", conn)
conn.close()

print(f"DataFrame loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nData types (all TEXT, as stored):")
print(df.dtypes)
print("\nFirst 5 rows:")
print(df.head())

DataFrame loaded: 550 rows x 10 columns

Data types (all TEXT, as stored):
customer_id      object
age              object
gender           object
country          object
channel          object
monthly_spend    object
clicks           object
impressions      object
conversions      object
signup_date      object
dtype: object

First 5 rows:
  customer_id age  gender   country    channel monthly_spend clicks  \
0    CUS_0000  22       F  Colombia  INSTAGRAM     $3,294.46    529   
1    CUS_0001  54  Female      Peru     TikTok          4351    260   
2    CUS_0002  48     Man     Chile     Google          2324    229   
3    CUS_0003  38             chile  Instagram     $1,314.41    322   
4    CUS_0004  38     Man        MX  Instagram   1259.65 USD    597   

  impressions conversions    signup_date  
0       43967          34     2024-04-01  
1        3572           4     2023-03-29  
2       20177          37  July 26, 2023  
3       23183           5     2024/10/01  
4       39995 

## 8. Save the raw DataFrame for the next notebook

In [8]:
df.to_csv(RAW_CSV, index=False)
print(f"✓ Saved to '{RAW_CSV}' — ready for cleaning")
print(f"  Rows: {len(df)} | Columns: {list(df.columns)}")

✓ Saved to '../data/raw/raw_customers.csv' — ready for cleaning
  Rows: 550 | Columns: ['customer_id', 'age', 'gender', 'country', 'channel', 'monthly_spend', 'clicks', 'impressions', 'conversions', 'signup_date']
